In [ ]:
server = 'obi'
logBoth = True
output_type = 'phoneme'


dataDir = '/victoriapvc/data/competitionData/'
if logBoth:
    dataSave = '/victoriapvc/data/ptDecoder_ctc_both'
else:
    dataSave = '/victoriapvc/data/ptDecoder_ctc'
        

        
if output_type == 'char':
    dataSave = f"{dataSave}_char"
    
if output_type == 'both':
    dataSave = f"{dataSave}_char_and_phoneme"
    

print(dataDir, dataSave)

/victoriapvc/data/competitionData/ /victoriapvc/data/ptDecoder_ctc_both


In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
import pickle

check_data = False

if check_data:

    if server == 'leia':
        
        with open('/home3/skaasyap/willett/data', 'rb') as f:
            data = pickle.load(f)
        #with open('/home3/skaasyap/willett/data_held_out_days', 'rb') as f:
        #    data_held_out = pickle.load(f)
        with open('/home3/skaasyap/willett/data_log_both', 'rb') as f:
            data_logged = pickle.load(f)
        with open('/home3/skaasyap/willett/data_log_both_held_out_days', 'rb') as f:
            data_logged_held_out = pickle.load(f)
            
        print(data['train'][0]['sentenceDat'][0].std(), data['train'][0]['sentenceDat'][0].mean())
        print(data_logged['train'][0]['sentenceDat'][0].std(), data_logged['train'][0]['sentenceDat'][0].mean())

    if server == 'obi':
        
        with open('/data/willett_data/ptDecoder_ctc', 'rb') as f:
            data = pickle.load(f)
        with open('/data/willett_data/ptDecoder_ctc_held_out_days', 'rb') as f:
            data_held_out = pickle.load(f)
        with open('/data/willett_data/ptDecoder_ctc_both', 'rb') as f:
            data_logged = pickle.load(f)
        with open('/data/willett_data/ptDecoder_ctc_both_held_out_days', 'rb') as f:
            data_logged_held_out = pickle.load(f)
        
#i = 10
#print(data['train'][i]['sentenceDat'][i].std(), data['train'][i]['sentenceDat'][i].mean())
#print(data_logged['train'][i]['sentenceDat'][i].std(), data_logged['train'][i]['sentenceDat'][i].mean())
#print(data_held_out['train'][i]['sentenceDat'][i].std(), data['train'][i]['sentenceDat'][i].mean())
#print(data_logged_held_out['train'][i]['sentenceDat'][i].std(), data_logged['train'][i]['sentenceDat'][i].mean())



In [ ]:
sessionNames = ['t12.2022.04.28',  't12.2022.05.26',  't12.2022.06.21',  't12.2022.07.21',  't12.2022.08.13',
't12.2022.05.05',  't12.2022.06.02',  't12.2022.06.23',  't12.2022.07.27',  't12.2022.08.18',
't12.2022.05.17',  't12.2022.06.07',  't12.2022.06.28',  't12.2022.07.29',  't12.2022.08.23',
't12.2022.05.19',  't12.2022.06.14',  't12.2022.07.05',  't12.2022.08.02',  't12.2022.08.25',
't12.2022.05.24',  't12.2022.06.16',  't12.2022.07.14',  't12.2022.08.11']
sessionNames.sort()

In [ ]:
import re 
from g2p_en import G2p
import numpy as np

g2p = G2p()
PHONE_DEF = [
    'AA', 'AE', 'AH', 'AO', 'AW',
    'AY', 'B',  'CH', 'D', 'DH',
    'EH', 'ER', 'EY', 'F', 'G',
    'HH', 'IH', 'IY', 'JH', 'K',
    'L', 'M', 'N', 'NG', 'OW',
    'OY', 'P', 'R', 'S', 'SH',
    'T', 'TH', 'UH', 'UW', 'V',
    'W', 'Y', 'Z', 'ZH'
]

PHONE_DEF_SIL = PHONE_DEF + ['SIL']

def phoneToId(p):
    return PHONE_DEF_SIL.index(p)
CHAR_VOCAB = [
    "<sp>",          # space token
    "!", ",", ".", "?", "'",   # punctuation (incl. apostrophe)
] + [chr(i) for i in range(ord('a'), ord('z') + 1)]  # 'a'..'z'

# Build mappings
_CHAR_TO_ID = {c: i for i, c in enumerate(CHAR_VOCAB)}
_ID_TO_CHAR = {i: c for c, i in _CHAR_TO_ID.items()}

# Convenience indices
SPACE_ID = _CHAR_TO_ID["<sp>"]

def charToId(c: str) -> int:
    """Map raw input char to ID, normalizing space and lowercase."""
    if c == " ":
        c = "<sp>"
    c = c.lower()
    return _CHAR_TO_ID[c]

def idToChar(i: int) -> str:
    return _ID_TO_CHAR[i]

In [5]:
def reorder_electrode_idxs(feats):
    
    '''
    In the original array, first 128 indices of feats are txcrossings, 
    second 128 features are spikeband power.
    Here what we do is rearrange feats so that:
        0-64: tx for inferior
        64-128: spikepow for inferior
        128-194: tx for superior
        194-256: spikepow for superior
    '''
    
    area_6v_superior = np.array([
    [62,  51,  43,  35,  94,  87,  79,  78],
    [60,  53,  41,  33,  95,  86,  77,  76],
    [63,  54,  47,  44,  93,  84,  75,  74],
    [58,  55,  48,  40,  92,  85,  73,  72],
    [59,  45,  46,  38,  91,  82,  71,  70],
    [61,  49,  42,  36,  90,  83,  69,  68],
    [56,  52,  39,  34,  89,  81,  67,  66],
    [57,  50,  37,  32,  88,  80,  65,  64]
    ])
    
    superior_combined_array = np.empty(area_6v_superior.size * 2, dtype=area_6v_superior.dtype)  
    
    # first 64 features are tx, second 64 are spikeband power
    superior_combined_array[:64] = area_6v_superior.ravel()
    superior_combined_array[64:] = area_6v_superior.ravel() + 128   

    area_6v_inferior = np.array([
        [125, 126, 112, 103,  31,  28,  11,  8],
        [123, 124, 110, 102,  29,  26,   9,  5],
        [121, 122, 109, 101,  27,  19,  18,  4],
        [119, 120, 108, 100,  25,  15,  12,  6],
        [117, 118, 107,  99,  23,  13,  10,  3],
        [115, 116, 106,  97,  21,  20,   7,  2],
        [113, 114, 105,  98,  17,  24,  14,  0],
        [127, 111, 104,  96,  30,  22,  16,  1]
    ])
    
    inferior_combined_array = np.empty(area_6v_inferior.size * 2, dtype=area_6v_inferior.dtype)  
    inferior_combined_array[:64] = area_6v_inferior.ravel()
    inferior_combined_array[64:] = area_6v_inferior.ravel() + 128   
    
    feats_reshaped = np.zeros_like(feats)
    
    feats_reshaped[:, 0:128] = feats[:, inferior_combined_array.ravel()]
    feats_reshaped[:, 128:] = feats[:, superior_combined_array.ravel()]
    
    return feats_reshaped
    

In [6]:
import scipy

def loadFeaturesAndNormalize(sessionPath, logBoth):
    
    dat = scipy.io.loadmat(sessionPath)

    input_features = []
    transcriptions = []
    frame_lens = []
    block_means = []
    block_stds = []
    n_trials = dat['sentenceText'].shape[0]

    #collect area 6v tx1 and spikePow features
    for i in range(n_trials):    
        #get time series of TX and spike power for this trial
        #first 128 columns = area 6v only
        if logBoth:
            tx_crossings = np.log1p(dat['tx1'][0, i][:, 0:128])
            log_pow = np.log(dat['spikePow'][0,i][:,0:128])
            features = np.concatenate([tx_crossings, log_pow], axis=1)
        else:
            features = np.concatenate([dat['tx1'][0,i][:,0:128], dat['spikePow'][0,i][:,0:128]], axis=1)

        sentence_len = features.shape[0]
        sentence = dat['sentenceText'][i].strip()

        input_features.append(features)
        transcriptions.append(sentence)
        frame_lens.append(sentence_len)

    #block-wise feature normalization
    blockNums = np.squeeze(dat['blockIdx'])
    blockList = np.unique(blockNums)
    blocks = []
    for b in range(len(blockList)):
        sentIdx = np.argwhere(blockNums==blockList[b])
        sentIdx = sentIdx[:,0].astype(np.int32)
        blocks.append(sentIdx)

    
    for b in range(len(blocks)):
        feats = reorder_electrode_idxs(np.concatenate(input_features[blocks[b][0]:(blocks[b][-1]+1)], axis=0))
        # feats = np.concatenate(input_features[blocks[b][0]:(blocks[b][-1]+1)], axis=0)

        feats_mean = np.mean(feats, axis=0, keepdims=True)
        feats_std = np.std(feats, axis=0, keepdims=True)
        for i in blocks[b]:
            input_features[i] = reorder_electrode_idxs(input_features[i])
            input_features[i] = (input_features[i] - feats_mean) / (feats_std + 1e-8)

    #convert to tfRecord file
    session_data = {
        'inputFeatures': input_features,
        'transcriptions': transcriptions,
        'frameLens': frame_lens, 
        'blockNums': blockNums
    }

    return session_data


In [7]:
import os
import re
import numpy as np

def getDataset(fileName, logBoth, output_type):
    session_data = loadFeaturesAndNormalize(fileName, logBoth)

    allDat = []
    trueSentences = []
    seq_char = []     # for characters (primary text)
    seq_phone = []    # for phonemes  (secondary text2, only when output_type == 'both')

    for x in range(len(session_data['inputFeatures'])):
        allDat.append(session_data['inputFeatures'][x])
        trueSentences.append(session_data['transcriptions'][x])

        # clean sentence
        thisTranscription = str(session_data['transcriptions'][x]).strip()
        thisTranscription = re.sub(r'[^a-zA-Z\- \']', '', thisTranscription)
        thisTranscription = thisTranscription.replace('--', '').lower()
        addInterWordSymbol = True
        maxSeqLen = 500

        # ----- build phoneme ids (used for 'phoneme' or 'both') -----
        if output_type in ('phoneme', 'both'):
            phonemes = []
            for p in g2p(thisTranscription):
                if addInterWordSymbol and p == ' ':
                    phonemes.append('SIL')
                p = re.sub(r'[0-9]', '', p)           # remove stress
                if re.match(r'[A-Z]+', p):            # keep only phoneme labels
                    phonemes.append(p)
            if addInterWordSymbol:
                phonemes.append('SIL')

            seqLenP = len(phonemes)
            seqIDsP = np.zeros([maxSeqLen], dtype=np.int32)
            seqIDsP[:seqLenP] = [phoneToId(p) + 1 for p in phonemes]

        # ----- build char ids (used for 'char' or 'both') -----
        if output_type in ('char', 'both'):
            txt = thisTranscription.replace('-', ' ')
            characters = []
            for c in txt:
                if addInterWordSymbol and c == ' ':
                    characters.append('<sp>')
                else:
                    characters.append(c)
            if addInterWordSymbol:
                characters.append('<sp>')

            seqLenC = len(characters)
            seqIDsC = np.zeros([maxSeqLen], dtype=np.int32)
            try:
                seqIDsC[:seqLenC] = [charToId(c) + 1 for c in characters]
            except Exception:
                print(txt)

        # ----- save according to output_type -----
        if output_type == 'phoneme':
            seq_char.append(seqIDsP)          # keep same key names as original ('text' holds the chosen output)
        elif output_type == 'char':
            seq_char.append(seqIDsC)
        elif output_type == 'both':
            seq_char.append(seqIDsC)          # 'text'  -> characters
            seq_phone.append(seqIDsP)         # 'text2' -> phonemes
        else:
            print("invalid output type, must be 'phoneme', 'char', or 'both'")
            return 0

    # ----- package dataset -----
    newDataset = {
        'sentenceDat': allDat,
        'transcriptions': trueSentences,
        'text': seq_char,                     # primary (chars for 'both')
        'blockNums': session_data['blockNums']
    }

    # lengths for primary text
    timeSeriesLens = [arr.shape[0] for arr in newDataset['sentenceDat']]
    textLens = []
    for arr in newDataset['text']:
        zeroIdx = np.argwhere(arr == 0)
        textLens.append(zeroIdx[0, 0] if zeroIdx.size else arr.shape[0])

    newDataset['timeSeriesLens'] = np.array(timeSeriesLens)
    newDataset['textLens'] = np.array(textLens)
    newDataset['textPerTime'] = newDataset['textLens'].astype(np.float32) / newDataset['timeSeriesLens'].astype(np.float32)

    # add phoneme versions when requested
    if output_type == 'both':
        newDataset['text2'] = seq_phone
        textLens2 = []
        for arr in newDataset['text2']:
            zeroIdx = np.argwhere(arr == 0)
            textLens2.append(zeroIdx[0, 0] if zeroIdx.size else arr.shape[0])
        newDataset['textLens2'] = np.array(textLens2)
        newDataset['textPerTime2'] = newDataset['textLens2'].astype(np.float32) / newDataset['timeSeriesLens'].astype(np.float32)

    return newDataset


In [10]:
# test contains the last block of each day

In [9]:
trainDatasets = []
testDatasets = []
competitionDatasets = []

for dayIdx in range(len(sessionNames)):
    
    trainDataset = getDataset(dataDir + '/train/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
    testDataset = getDataset(dataDir + '/test/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)

    trainDatasets.append(trainDataset)
    testDatasets.append(testDataset)

    if os.path.exists(dataDir + '/competitionHoldOut/' + sessionNames[dayIdx] + '.mat'):
        dataset = getDataset(dataDir + '/competitionHoldOut/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
        competitionDatasets.append(dataset)
        
        
            

KeyboardInterrupt: 

In [10]:
allDatasets = {}
allDatasets['train'] = trainDatasets
allDatasets['test'] = testDatasets
allDatasets['competition'] = competitionDatasets


with open(dataSave, 'wb') as handle:
    pickle.dump(allDatasets, handle)

In [8]:
def return_train_validation_blocks(unique_blocks, K, fold_number):
    
    """
    list unique_blocks: unique blocks in that day
    int K: the number of folds
    fold_number: current fold
    
    Returns the blocks that should be used for training and validation. 
    
    * This function assumes the last block was already used for validation, since this was the default competition settings.
    """
    
    n_blocks_for_val = len(unique_blocks) - 1
    n_blocks_per_fold = np.ceil(n_blocks_for_val / K)
    
    start_idx = int(fold_number*n_blocks_per_fold)
    end_idx = int(min(n_blocks_for_val, (fold_number+1)*n_blocks_per_fold))
        
    blocks_for_val = unique_blocks[start_idx:end_idx]
    
    return np.setdiff1d(unique_blocks, blocks_for_val), blocks_for_val

def partition_by_blocks(dataset, train_blocks, test_blocks):
    
    """Split dataset into train and test portions using blockNums."""
    train_set, test_set = {}, {}
    
    for key, values in dataset.items():
        # blockNums is special: we need it to decide the split
        if key == "blockNums":
            continue

        # Assume dataset["blockNums"] is aligned with all other keys
        train_set[key] = [v for v, b in zip(values, dataset["blockNums"]) if b in train_blocks]
        test_set[key]  = [v for v, b in zip(values, dataset["blockNums"]) if b in test_blocks]

    # Also save the blockNums
    train_set["blockNums"] = [b for b in dataset["blockNums"] if b in train_blocks]
    test_set["blockNums"]  = [b for b in dataset["blockNums"] if b in test_blocks]

    return train_set, test_set

In [11]:
import pickle
trainSentences_arr = []

for fold_number in [0,1,2,3,4]:
    
    trainSentences = 0
    trainDatasets = []
    testDatasets = []
    competitionDatasets = []
        
    for dayIdx in range(len(sessionNames)):
        
        trainDataset = getDataset(dataDir + '/train/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
        testDataset = getDataset(dataDir + '/test/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
        
        combined = {}
        
        # combine train and validation data. 
        for key in trainDataset.keys():
            if type(trainDataset[key]) == np.ndarray:
                combined[key] = np.concatenate((trainDataset[key], testDataset[key]))
            else:
                combined[key] = trainDataset[key] + testDataset[key]
                
        block_numbers = np.unique(combined['blockNums'])
                
        train_blocks, validation_blocks = return_train_validation_blocks(block_numbers, K=5, fold_number=fold_number)
        
        trainDataset, testDataset = partition_by_blocks(combined, train_blocks, validation_blocks)
        
        trainSentences += len(trainDataset['sentenceDat'])

        trainDatasets.append(trainDataset)
        testDatasets.append(testDataset)

        if os.path.exists(dataDir + '/competitionHoldOut/' + sessionNames[dayIdx] + '.mat'):
            dataset = getDataset(dataDir + '/competitionHoldOut/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
            competitionDatasets.append(dataset)
            
    trainSentences_arr.append(trainSentences)
    
    print(f"Fold number {trainSentences}")
    
    allDatasets = {}
    allDatasets['train'] = trainDatasets
    allDatasets['test'] = testDatasets
    allDatasets['competition'] = competitionDatasets
    
    dataSave_fold = f"{dataSave}_fold_number_{fold_number}"

    with open(dataSave_fold, 'wb') as handle:
        pickle.dump(allDatasets, handle)
        
    print("Saved data")

KeyboardInterrupt: 

In [161]:
dataSave

'/data/willett_data/ptDecoder_ctc_both'

In [8]:
import pickle


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

data = trainDatasets[0]['sentenceDat'][10].T
np.save('figure1_neural_data', data)

plt.figure(figsize=(5, 3))
plt.imshow(trainDatasets[0]['sentenceDat'][10].T[:128, 0:200], clim=[-1, 1], alpha=0.4, cmap='bone')
plt.xlabel("Time (s)", fontsize=16)
plt.ylabel("Features", fontsize=16)
plt.tick_params(axis='x', length=0)
plt.xticks([0, 200], [0, 4])
plt.yticks([])


# Transparent rectangle
rect = patches.Rectangle((130, -5), 32, 138,
                         facecolor='none', edgecolor='black', linewidth=2, clip_on=False)
plt.gca().add_patch(rect)


# Adjust y-limits to show full rectangle
plt.ylim(-10, 135)

# Despine all axes
for spine in plt.gca().spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.figure(figsize=(5, 3))
plt.imshow(trainDatasets[0]['sentenceDat'][10].T[:128, 0:200], clim=[-1, 1], alpha=0.4, cmap='bone')
plt.xlabel("Time (s)", fontsize=16)
plt.ylabel("")
plt.tick_params(axis='x', length=0)
plt.xticks([0, 200], [0, 4])
plt.yticks([])


# Transparent rectangle
rect = patches.Rectangle((134, -5), 32, 138,
                         facecolor='none', edgecolor='black', linewidth=2, clip_on=False)
plt.gca().add_patch(rect)


# Adjust y-limits to show full rectangle
plt.ylim(-10, 135)

# Despine all axes
for spine in plt.gca().spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

data = trainDatasets[0]['sentenceDat'][10].T[:64, :260]


fig, ax = plt.subplots(figsize=(5,4))
ax.imshow(data, vmin=-1, vmax=1, origin='lower', alpha=0.6)
ax.set_yticks([])
centers = [60, 130, 200]
rect_width = 20
rect_height = data.shape[0]  # full height of the electrode axis

# Add rectangles
y_start = -0.5  # start slightly below the image
rect_height = data.shape[0] + 1  # extend slightly beyond top

for x_center in centers:
    rect = patches.Rectangle((x_center - rect_width / 2, y_start), rect_width, rect_height,
                             linewidth=0, facecolor='white', alpha=1.0)
    ax.add_patch(rect)
sns.despine(trim=True, top=True, right=True, left=True, bottom=True)

# Set x-ticks at the edges of each patch
ax.set_xticks([])
tick_positions = [0, 50, 70, 120, 140, 190, 210, 260]
tick_positions = [2, 48, 72, 118, 142, 188, 212, 258]
ax.set_xticks(tick_positions)
tick_labels = np.array([0, 50, 50, 100, 100, 150, 150, 200])/10 # 4 labels for the 4 patches
ax.set_xticklabels([str(int(lbl)) for lbl in tick_labels])
ax.tick_params(axis='x', length=0)  # Set tick mark length to 0 on x-axis
ax.tick_params(axis='x', labelsize=12)  

ax.set_xlabel("Timesteps", fontsize=16)

plt.ylim(-0.5, data.shape[0] + 0.5)
plt.tight_layout()
plt.show()



fig, ax = plt.subplots(figsize=(5,4))
ax.imshow(data, vmin=-1, vmax=1, origin='lower', alpha=0.6)
ax.set_yticks([])
centers = [60, 130, 200]
rect_width = 20
rect_height = data.shape[0]  # full height of the electrode axis

# Add rectangles
y_start = -0.5  # start slightly below the image
rect_height = data.shape[0] + 1  # extend slightly beyond top

for x_center in centers:
    rect = patches.Rectangle((x_center - rect_width / 2, y_start), rect_width, rect_height,
                             linewidth=0, facecolor='white', alpha=1.0)
    ax.add_patch(rect)
sns.despine(trim=True, top=True, right=True, left=True, bottom=True)

# Set x-ticks at the edges of each patch
ax.set_xticks([])
tick_positions = [0, 50, 70, 120, 140, 190, 210, 260]
tick_positions = [2, 48, 72, 118, 142, 188, 212, 258]
ax.set_xticks(tick_positions)
tick_labels = [0, 32, 4, 36, 8, 40, 12, 44] # 4 labels for the 4 patches
ax.set_xticklabels([str(int(lbl)) for lbl in tick_labels])
ax.tick_params(axis='x', length=0)  # Set tick mark length to 0 on x-axis
ax.tick_params(axis='x', labelsize=12)  

ax.set_xlabel("Timesteps", fontsize=16)

plt.ylim(-0.5, data.shape[0] + 0.5)
plt.tight_layout()
plt.show()

In [15]:
def combine_dicts(dict1, dict2):
    
    combined_dict = {}
    for key in dict1.keys():
        if key in ['sentenceDat', 'transcriptions', 'phonemes']: 
            combined_dict[key] = dict1[key] + dict2[key]
        else:
            combined_dict[key] = np.concatenate([dict1[key], dict2[key]])
        
    return combined_dict

In [ ]:
sessionNames

# Held out days testing

## we put last three days as test set

In [ ]:
def combine_dicts(dict1, dict2):
    
    combined_dict = {}
    for key in dict1.keys():
        if key in ['sentenceDat', 'transcriptions', 'phonemes']: 
            combined_dict[key] = dict1[key] + dict2[key]
        else:
            combined_dict[key] = np.concatenate([dict1[key], dict2[key]])
        
    return combined_dict

In [33]:
# test set
output_type = 'both'
realTestDatasets = []
for dayIdx in range(len(sessionNames)-3, len(sessionNames)):
    
       
        print(f"Placing {sessionNames[dayIdx]} in test")
        
        # put both train and test data for held-out days into "test (really val)".
        testDataset1 = getDataset(dataDir + '/train/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
        testDataset2 = getDataset(dataDir + '/test/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
        testDataset = combine_dicts(testDataset1, testDataset2)
        realTestDatasets.append(testDataset)
        
crossdaytestDatasets = {}
crossdaytestDatasets['test'] = realTestDatasets
crossdaytestDatasets['train'] = realTestDatasets # placeholder, not used
dataSave_held_out_days =f"{dataSave}_last3days"
with open(dataSave_held_out_days, 'wb') as handle:
    pickle.dump(crossdaytestDatasets, handle)


Placing t12.2022.08.18 in test


Placing t12.2022.08.23 in test
Placing t12.2022.08.25 in test


In [28]:
sessionNames

['t12.2022.04.28',
 't12.2022.05.05',
 't12.2022.05.17',
 't12.2022.05.19',
 't12.2022.05.24',
 't12.2022.05.26',
 't12.2022.06.02',
 't12.2022.06.07',
 't12.2022.06.14',
 't12.2022.06.16',
 't12.2022.06.21',
 't12.2022.06.23',
 't12.2022.06.28',
 't12.2022.07.05',
 't12.2022.07.14',
 't12.2022.07.21',
 't12.2022.07.27',
 't12.2022.07.29',
 't12.2022.08.02',
 't12.2022.08.11',
 't12.2022.08.13',
 't12.2022.08.18',
 't12.2022.08.23',
 't12.2022.08.25']

In [30]:
logBoth

True

In [31]:
output_type

'phoneme'

In [37]:
realTestDatasets = []
output_type = 'both'
skip_days = ['07.29']
for dayIdx in range(16, 24):
    if sessionNames[dayIdx][-5:] in skip_days:
        continue
    print(f"Placing {sessionNames[dayIdx]} in test")

    # put both train and test data for held-out days into "test (really val)".
    testDataset1 = getDataset(dataDir + '/train/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
    testDataset2 = getDataset(dataDir + '/test/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
    testDataset = combine_dicts(testDataset1, testDataset2)
    realTestDatasets.append(testDataset)
        
crossdaytestDatasets = {}
crossdaytestDatasets['test'] = realTestDatasets
crossdaytestDatasets['train'] = realTestDatasets # placeholder, not used
dataSave_held_out_days =f"{dataSave}_last7days"
with open(dataSave_held_out_days, 'wb') as handle:
    pickle.dump(crossdaytestDatasets, handle)

Placing t12.2022.07.27 in test
Placing t12.2022.08.02 in test
Placing t12.2022.08.11 in test
Placing t12.2022.08.13 in test
Placing t12.2022.08.18 in test
Placing t12.2022.08.23 in test
Placing t12.2022.08.25 in test


In [ ]:
competitionDays = []
for dayIdx in range(len(sessionNames)):
    if os.path.exists(dataDir + '/competitionHoldOut/' + sessionNames[dayIdx] + '.mat'):
        competitionDays.append(dayIdx)
print(competitionDays)

for cd in competitionDays:
    print(sessionNames[cd])

['04.28', '05.05', '05.17','05.19', '05.24', 
 '05.26', '06.02', '06.07', '06.14', '06.16', 
 '06.21', '06.23', '06.28', '07.05', '07.14', 
 '07.21', '07.27', '08.02', '08.11','08.13',]

no held out days: ['04.28', '05.05', '05.17','05.19', '06.23'
 '07.29', '08.18', '08.23', '08.25']


In [ ]:
condition = 7
if condition == 0: # 15 vs 5
    evaluation_days = ['07.21', '07.27', '08.02', '08.11', '08.13'] 
    skip_days = ['07.29']
if condition == 1: # 12 vs 8
    evaluation_days = ['06.28', '07.05', '07.14', '07.21', '07.27', '08.02', '08.11', '08.13'] 
    skip_days = ['07.29']
if condition == 2: # interleaved 15 vs 5
    evaluation_days = ['04.28', '05.17', '06.14', '07.14', '08.13']
    skip_days = ['07.29']
    # test_days = ['08.18', '08.23', '08.25']
if condition == 3: # interleaved eval no april
    evaluation_days = ['05.26', '06.14', '07.14','07.27', '08.13']   
    skip_days = ['07.29']
    # test_days = ['08.18', '08.23', '08.25']
if condition == 4: # 8 vs 8, 4 gaps '06.14', '06.16', '06.21', '06.23'
    evaluation_days = ['06.28', '07.05', '07.14', '07.21', '07.27', '08.02', '08.11', '08.13'] 
    skip_days = ['06.14', '06.16', '06.21', '06.23', '07.29']
if condition == 5: # 12 vs 4, 4 gaps '06.28', '07.05', '07.14', '07.21'
    evaluation_days = ['07.27', '08.02', '08.11', '08.13'] 
    skip_days = ['06.28', '07.05', '07.14', '07.21', '07.29']
if condition == 6: # 1 vs 1  train 07.21 eval 07.27
    evaluation_days = ['07.27'] 
    skip_days = ['04.28', '05.05', '05.17','05.19', '05.24', '05.26', '06.02', '06.07', '06.14', '06.16', '06.21', '06.23', '06.28', '07.05', '07.14', '08.02', '08.11','08.13',]
if condition == 7: # 15, '07.21',1 gap, 4 eval
    evaluation_days = [ '07.27', '08.02', '08.11', '08.13'] 
    skip_days = ['07.29', '07.21']
if condition == 8: # 18 train, 2 val
    evaluation_days = ['08.11', '08.13'] 
    skip_days = ['07.29']
if condition == 9: # 18 train, 1 gap 1 val
    evaluation_days = ['08.11', '08.13'] 
    skip_days = ['07.29', '08.02']
if condition == 10: # 19 train, 1 val
    evaluation_days = ['08.13'] 
    skip_days = ['07.29']
if condition == 11: # 17 train, 3 val
    evaluation_days = ['08.02', '08.11', '08.13'] 
    skip_days = ['07.29']
if condition == 12: # 17 train, 1 gap 2 val
    evaluation_days = ['08.11', '08.13'] 
    skip_days = ['07.29', '08.02']
if condition == 13: # 16 train, 4 val
    evaluation_days = ['07.27', '08.02', '08.11', '08.13'] 
    skip_days = ['07.29']
if condition == 14: # 16 train, 1 gap 3 val
    evaluation_days = ['08.02', '08.11', '08.13'] 
    skip_days = ['07.29', '07.27']
    
    


In [12]:
trainDatasets = []
testDatasets = []
competitionDatasets = []
output_type = 'both'


trainCount = 0
evalCount = 0
print(logBoth)
for dayIdx in range(len(sessionNames[:-3])):
    
    if sessionNames[dayIdx][-5:] in skip_days:
        print("Skipping day: ", sessionNames[dayIdx][-5:])
        continue
        
    if sessionNames[dayIdx][-5:] in evaluation_days:
        
        evalCount += 1
        
        print(f"Placing {sessionNames[dayIdx]} in val")
        
        # put both train and test data for held-out days into "test (really val)".
        testDataset1 = getDataset(dataDir + '/train/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
        testDataset2 = getDataset(dataDir + '/test/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
        testDataset = combine_dicts(testDataset1, testDataset2)
        testDatasets.append(testDataset)
        
        # only testing on these held-out days now. 
        if os.path.exists(dataDir + '/competitionHoldOut/' + sessionNames[dayIdx] + '.mat'):
            dataset = getDataset(dataDir + '/competitionHoldOut/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
            competitionDatasets.append(dataset)
            print(f"Placing {sessionNames[dayIdx]} in comp")
            
    if sessionNames[dayIdx][-5:] not in evaluation_days and sessionNames[dayIdx][-5:] not in skip_days:
        
        trainCount += 1
        
        print(f"Train {sessionNames[dayIdx]} in train")
        
        # put both train and test data for non held-out days into train
        trainDataset1 = getDataset(dataDir + '/train/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
        trainDataset2 = getDataset(dataDir + '/test/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
        trainDataset = combine_dicts(trainDataset1, trainDataset2)
        trainDatasets.append(trainDataset)
        
        
print("Training days: ", trainCount)
print("Eval days: ", evalCount)

True
Train t12.2022.04.28 in train
Train t12.2022.05.05 in train
Train t12.2022.05.17 in train
Train t12.2022.05.19 in train
Train t12.2022.05.24 in train
Train t12.2022.05.26 in train
Train t12.2022.06.02 in train
Train t12.2022.06.07 in train
Train t12.2022.06.14 in train
Train t12.2022.06.16 in train
Train t12.2022.06.21 in train
Train t12.2022.06.23 in train
Train t12.2022.06.28 in train
Train t12.2022.07.05 in train
Train t12.2022.07.14 in train
Skipping day:  07.21
Placing t12.2022.07.27 in val
Placing t12.2022.07.27 in comp
Skipping day:  07.29
Placing t12.2022.08.02 in val
Placing t12.2022.08.02 in comp
Placing t12.2022.08.11 in val
Placing t12.2022.08.11 in comp
Placing t12.2022.08.13 in val
Placing t12.2022.08.13 in comp
Training days:  15
Eval days:  4


In [69]:
import pickle
allDatasets = {}
allDatasets['train'] = trainDatasets
allDatasets['test'] = testDatasets
allDatasets['competition'] = competitionDatasets

In [ ]:
condition = 4
if condition == 0: 
    dataSave_held_out_days = f"{dataSave}_held_out_days"
if condition == 1:
    dataSave_held_out_days = f"{dataSave}_held_out_days_big_0"
if condition == 2:
    dataSave_held_out_days = f"{dataSave}_held_out_days_interleaved"
if condition == 3:
    dataSave_held_out_days = f"{dataSave}_held_out_days_interleaved_no_april"
if condition == 4:
    dataSave_held_out_days = f"{dataSave}_held_out_days_big_1"
if condition == 5:
    dataSave_held_out_days = f"{dataSave}_held_out_days_big_2"
if condition == 6:
    dataSave_held_out_days = f"{dataSave}_held_out_days_1_1"
if condition == 7:
    dataSave_held_out_days = f"{dataSave}_held_out_days_15_1_4"
if condition == 8:
    dataSave_held_out_days = f"{dataSave}_held_out_days_18_2"
if condition == 9:
    dataSave_held_out_days = f"{dataSave}_held_out_days_18_1_1"
if condition == 10:
    dataSave_held_out_days = f"{dataSave}_held_out_days_19_1"
    

In [ ]:
# with open(dataSave_held_out_days, 'rb') as handle:
#     allDatasets = pickle.load(handle)
#     print(allDatasets.keys())
# print(len(allDatasets['competition']))
# print(len(allDatasets['train']))
# print(len(allDatasets['test']))
# print(len(allDatasets['skipped']))


8
8
8
4


In [74]:
print(allDatasets.keys())

dict_keys(['train', 'test', 'competition', 'competition_source', 'skipped'])


# add source day that in orginal heldout set as 'competition_source'

In [72]:

if 'competition_source' not in allDatasets.keys():
    competitionDatasets = []
    competitionDatasets_source = []

    trainCount = 0
    evalCount = 0
    print(logBoth)
    for dayIdx in range(len(sessionNames[:-3])):
        
        if sessionNames[dayIdx][-5:] in skip_days:
        
            continue
            
        if sessionNames[dayIdx][-5:] in evaluation_days:
            
            continue
                
        else:
            # only testing on these held-out days now. 
            if os.path.exists(dataDir + '/competitionHoldOut/' + sessionNames[dayIdx] + '.mat'):
                evalCount +=1
                print(f"Placing {sessionNames[dayIdx]} in competition_source")
                dataset = getDataset(dataDir + '/competitionHoldOut/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
                competitionDatasets_source.append(dataset)
            
            
    print("Training days: ", trainCount)
    print("competition_source days: ", evalCount)

    allDatasets['competition_source'] = competitionDatasets_source 

True
Placing t12.2022.05.24 in competition_source
Placing t12.2022.05.26 in competition_source
Placing t12.2022.06.02 in competition_source
Placing t12.2022.06.07 in competition_source
Placing t12.2022.06.14 in competition_source
Placing t12.2022.06.16 in competition_source
Placing t12.2022.06.21 in competition_source
Training days:  0
competition_source days:  7


# add skipped days  as 'skipped'

In [73]:
if 'skipped' not in allDatasets.keys() and skip_days != []:
    skippedDatasets = []
    evalCount = 0
    for dayIdx in range(len(sessionNames[:-3])):
        
        if sessionNames[dayIdx][-5:] in skip_days and sessionNames[dayIdx][-5:] != '07.29':
            
            evalCount += 1
            
            print(f"Placing {sessionNames[dayIdx]} in skipped")
            
            # put both train and test data for held-out days into "test (really val)".
            skippedDataset1 = getDataset(dataDir + '/train/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
            skippedDataset2 = getDataset(dataDir + '/test/' + sessionNames[dayIdx] + '.mat', logBoth, output_type=output_type)
            skippedDataset = combine_dicts(skippedDataset1, skippedDataset2)
            skippedDatasets.append(skippedDataset)
            
                
    print("skipped days: ", evalCount)
    allDatasets['skipped'] = skippedDatasets 

Placing t12.2022.06.28 in skipped
Placing t12.2022.07.05 in skipped
Placing t12.2022.07.14 in skipped
Placing t12.2022.07.21 in skipped
skipped days:  4


In [51]:
print(allDatasets.keys())

dict_keys(['train', 'test', 'competition', 'competition_source', 'skipped'])


In [75]:
len(allDatasets['train'])

12

In [76]:
# check if all datasets are in the dictionary

print(allDatasets['competition_source'][0]['transcriptions'])
print(allDatasets['skipped'][0]['transcriptions'] if 'skipped' in allDatasets.keys() else 'no skipped days')

['HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT', 'HELD OUT']
['After that it was the interior stairs

In [77]:
with open(dataSave_held_out_days, 'wb') as handle:
    pickle.dump(allDatasets, handle)
    print("saved ", dataSave_held_out_days)

saved  /victoriapvc/data/ptDecoder_ctc_both_held_out_days_big_2


# draft

In [13]:
print(allDatasets.keys())

dict_keys(['train', 'test', 'competition', 'competition_source'])


In [15]:
allDatasets['competition_source']

[{'sentenceDat': [array([[ 0.3084071 , -0.79358804,  0.89417267, ...,  0.29495943,
            0.41590273,  0.17818837],
          [ 1.0933044 ,  0.7599403 , -0.74480677, ..., -0.45595223,
           -0.19506285,  1.9850049 ],
          [ 1.0933044 ,  0.7599403 ,  1.8521677 , ..., -0.70536304,
            1.0443598 ,  0.7169538 ],
          ...,
          [-1.0344293 ,  0.7599403 ,  0.89417267, ...,  0.29255143,
            0.65928566,  2.3198376 ],
          [-1.0344293 ,  0.7599403 ,  0.89417267, ...,  0.14619987,
            0.95971686, -0.01027176],
          [ 0.3084071 , -0.79358804,  0.89417267, ..., -0.7439089 ,
            1.8044212 , -0.0431204 ]], dtype=float32),
   array([[-1.0344293 ,  0.7599403 ,  0.89417267, ..., -1.3911268 ,
           -0.71661514, -0.60999   ],
          [-1.0344293 ,  1.6679885 ,  0.89417267, ...,  0.36520427,
            1.0761727 , -1.3060375 ],
          [ 0.3084071 ,  0.7599403 ,  1.8521677 , ..., -1.3965132 ,
           -1.6807902 , -2.1664314 ],

In [16]:
dataSave_held_out_days

'/victoriapvc/data/ptDecoder_ctc_both_held_out_days_big_1'

In [14]:

with open(dataSave_held_out_days, 'wb') as handle:
    pickle.dump(allDatasets, handle)

# add skipped days

In [12]:
trainDatasets = []
testDatasets = []
competitionDatasets = []
output_type = 'both'
condition = 4
skip_days = ['07.29']
if condition == 0:
    evaluation_days = ['07.21', '07.27', '08.02', '08.11', '08.13'] 
if condition == 1:
    evaluation_days = ['06.28', '07.05', '07.14', '07.21', '07.27', '08.02', '08.11', '08.13'] 
if condition == 2:
    evaluation_days = ['04.28', '05.17', '06.14', '07.14', '08.13']
    test_days = ['08.18', '08.23', '08.25']
if condition == 3:
    evaluation_days = ['05.26', '06.14', '07.14','07.27', '08.13']   
    test_days = ['08.18', '08.23', '08.25']
if condition == 4:
    evaluation_days = ['06.28', '07.05', '07.14', '07.21', '07.27', '08.02', '08.11', '08.13'] 
    skip_days = ['06.14', '06.16', '06.21', '06.23', '07.29']
if condition == 5:
    evaluation_days = ['07.27', '08.02', '08.11', '08.13'] 
    skip_days = ['06.28', '07.05', '07.14', '07.21', '07.29']
if condition == 6:
    evaluation_days = ['07.27', '08.02', '08.11', '08.13'] 
    skip_days = ['04.28', '05.05', '06.28', '07.05', '07.14', '07.21', '07.29']
if condition == 7: 
    evaluation_days = ['07.27', '08.02', '08.11', '08.13'] 
    skip_days = ['04.28', '05.05', '06.28', '07.05', '07.14', '07.21', '07.29']

Placing t12.2022.06.14 in skipped
Placing t12.2022.06.16 in skipped
Placing t12.2022.06.21 in skipped
Placing t12.2022.06.23 in skipped
Eval days:  4


In [17]:
skippedDatasets[0]['transcriptions']

['So that was a wonderful vacation.',
 'I think those are pretty exciting.',
 "We've had one as long as I can remember.",
 'So I can go out and go to the bar this late?',
 "I guess it's a couple of years old.",
 "Not that great or didn't do that great anyway.",
 'So I got under there and messed with the voice.',
 'And things going on.',
 'I made a decision.',
 'Subscriptions to this.',
 'I have seen a big change.',
 'Every time something new came up.',
 'Trainable disposition.',
 'You should make yourself a promise.',
 "Who'll be inconvenienced by the closures?",
 'It must be so nice for you.',
 'I need a hole there.',
 'Those are the two I like the best.',
 'What kind of dog do you have.',
 'But that was a really fun.',
 "That sounds I wouldn't buy something else.",
 'Atlanta is horrible.',
 "You're the farthest person I've ever talked to.",
 'He was infatuated by that skunk.',
 'Coconut oil as a home remedy.',
 'This is one step above it.',
 'Well probably something like that.',
 'I 

In [18]:
if condition == 0: 
    dataSave_held_out_days = f"{dataSave}_held_out_days"
if condition == 1:
    dataSave_held_out_days = f"{dataSave}_held_out_days_big_0"
if condition == 2:
    dataSave_held_out_days = f"{dataSave}_held_out_days_interleaved"
if condition == 3:
    dataSave_held_out_days = f"{dataSave}_held_out_days_interleaved_no_april"
if condition == 4:
    dataSave_held_out_days = f"{dataSave}_held_out_days_big_1"

with open(dataSave_held_out_days, 'rb') as handle:
    allDatasets = pickle.load(handle)
    print(allDatasets.keys())
allDatasets['skipped'] = skippedDatasets

dict_keys(['train', 'test', 'competition', 'competition_source', 'skipped'])


In [20]:
skippedDatasets[0]['text']

array([[25, 21,  1, ...,  0,  0,  0],
       [15,  1, 26, ...,  0,  0,  0],
       [29, 11,  6, ...,  0,  0,  0],
       ...,
       [26, 14, 11, ...,  0,  0,  0],
       [26, 14,  7, ...,  0,  0,  0],
       [23, 27, 15, ...,  0,  0,  0]], dtype=int32)

In [21]:

with open(dataSave_held_out_days, 'wb') as handle:
    pickle.dump(allDatasets, handle)

In [22]:
with open(dataSave_held_out_days, 'rb') as handle:
    allDatasets = pickle.load(handle)
    print(allDatasets.keys())
allDatasets['skipped'] = skippedDatasets
skippedDatasets[0]['text']

dict_keys(['train', 'test', 'competition', 'competition_source', 'skipped'])


array([[25, 21,  1, ...,  0,  0,  0],
       [15,  1, 26, ...,  0,  0,  0],
       [29, 11,  6, ...,  0,  0,  0],
       ...,
       [26, 14, 11, ...,  0,  0,  0],
       [26, 14,  7, ...,  0,  0,  0],
       [23, 27, 15, ...,  0,  0,  0]], dtype=int32)

In [19]:
print(dataSave)

/victoriapvc/data/ptDecoder_ctc_both


In [15]:
import pickle
allDatasets = {}
allDatasets['train'] = trainDatasets
allDatasets['test'] = testDatasets
allDatasets['competition'] = competitionDatasets

if condition == 0: 
    dataSave_held_out_days = f"{dataSave}_held_out_days"
if condition == 1:
    dataSave_held_out_days = f"{dataSave}_held_out_days_big_0"
if condition == 2:
    dataSave_held_out_days = f"{dataSave}_held_out_days_interleaved"
if condition == 3:
    dataSave_held_out_days = f"{dataSave}_held_out_days_interleaved_no_april"
if condition == 4:
    dataSave_held_out_days = f"{dataSave}_held_out_days_big_1"
    
print(dataSave_held_out_days)
with open(dataSave_held_out_days, 'wb') as handle:
    pickle.dump(allDatasets, handle)

/victoriapvc/data/ptDecoder_ctc_both_held_out_days_big_1


In [ ]:
print("max day held out 0 ", 24 - 3 - 6 - 1)
print("max day held out 1 ", 24 -9 - 5 - 1)
print("max day held out 2 ", 24 - 15 - 5 - 1)

In [ ]:
import numpy as np
evaluation_days = ['07.21', '07.27', '08.02', 
                 '08.11', '08.13', '07.29', '07.14', '07.05', '06.28', '06.23', '06.21', '06.16', 
                 '06.14', '06.07', '06.02', '05.26', '05.24', '05.19', '05.17', '05.05', '04.28']

sessionNames_days = np.array([x[-5:] for x in sessionNames])
# skipping the last 3 days because they are not historical data. 
for eval_day in evaluation_days:
    
    print(np.argwhere(eval_day==sessionNames_days)[0])
        

In [ ]:
trainDatasets = []
testDatasets = []
competitionDatasets = []

held_out_run_set = 2

all_training_days = sessionNames[:-3] # first four days will be included in all conditions

all_training_days = [x[-5:] for x in all_training_days]

# remove the one silent speaking day and the one equipment failure day
all_training_days.remove('06.23')
all_training_days.remove('07.29')

# in the first condition, include up to the last 5 competition days in the training data
if held_out_run_set == 0:

    # one extra evaluation day here
    evaluation_days = ['07.21', '07.27', '08.02', '08.11', '08.13'] 
    loop_until = -3
    
elif held_out_run_set == 1:
    
    evaluation_days = ['06.16', '06.21', '06.28', '07.05', '07.14']
    
elif held_out_run_set == 2:
    
    evaluation_days = ['05.24', '05.26', '06.02', '06.07', '06.14']

train_days_count = 0
eval_days_count = 0
# skipping the last 3 days because they are not historical data. 
for dayIdx in range(len(sessionNames[:-3])):

    if sessionNames[dayIdx][-5:] in evaluation_days:
        
        eval_days_count += 1

        print("EVAL", sessionNames[dayIdx])
            
        # put both train and test data for held-out days into "test (really val)".
        testDataset1 = getDataset(dataDir + '/train/' + sessionNames[dayIdx] + '.mat', logBoth)
        testDataset2 = getDataset(dataDir + '/test/' + sessionNames[dayIdx] + '.mat', logBoth)
        
        testDataset = combine_dicts(testDataset1, testDataset2)
        testDatasets.append(testDataset)
        
        # only testing on these held-out days now. 
        if os.path.exists(dataDir + '/competitionHoldOut/' + sessionNames[dayIdx] + '.mat'):
            dataset = getDataset(dataDir + '/competitionHoldOut/' + sessionNames[dayIdx] + '.mat', logBoth)
            competitionDatasets.append(dataset)
            
    elif sessionNames[dayIdx][-5:] in all_training_days:
        
        print("TRAIN", sessionNames[dayIdx])
        train_days_count += 1
        
        # put both train and test data for non held-out days into train
        trainDataset1 = getDataset(dataDir + '/train/' + sessionNames[dayIdx] + '.mat', logBoth)
        trainDataset2 = getDataset(dataDir + '/test/' + sessionNames[dayIdx] + '.mat', logBoth)
        trainDataset = combine_dicts(trainDataset1, trainDataset2)
        trainDatasets.append(trainDataset)
        
print(eval_days_count, train_days_count)
        

In [ ]:
from neural_decoder.dataset import getDatasetLoaders
datasetPath = '/victoriapvc/data/ptDecoder_ctc_both_held_out_days'
trainLoader, testLoader, loadedData = getDatasetLoaders(
        datasetPath,
        8,
        [],
        False,
    )

ImportError: cannot import name 'getDatasetLoaders' from 'neural_decoder.dataset' (/victoriapvc/repos/brain2text/src/neural_decoder/dataset.py)